# Startup Investment Attribution - Understand Outreach Effort's Effect

**This sample notebook aims to show the application of using SynapseML's DoubleMLEstimator for inferring causality using observational data.**

A startup that sells software would like to know whether its outreach efforts were successful in attracting new customers or boosting consumption among existing customers. In other words, they would like to learn the treatment effect of each investment on customers' software usage.

In an ideal world, the startup would run several randomized experiments where each customer would receive a random assortment of investments. However, this can be logistically prohibitive or strategically unsound: the startup might not have the resources to design such experiments or they might not want to risk losing out on big opportunities due to lack of incentives.

In this customer scenario walkthrough, we show how SynapseML causal package can use historical investment data to learn the investment effect.

## Background
In this scenario, a startup that sells software provides discounts incentives to its customer. A customer might be given or not.

The startup has historical data on these investments for 2,000 customers, as well as how much revenue these customers generated in the year after the investments were made. They would like to use this data to learn the optimal incentive policy for each existing or new customer in order to maximize the return on investment (ROI).

The startup faces a challenge:  the dataset is biased because historically the larger customers received the most incentives. Thus, they need a causal model that can remove the bias.

## Data
The data* contains ~2,000 customers and is comprised of:

* Customer features: details about the industry, size, revenue, and technology profile of each customer.
* Interventions: information about which incentive was given to a customer.
* Outcome: the amount of product the customer bought in the year after the incentives were given.


| Feature Name    | Type | Details                                                                                                                                     |
|-----------------|------|---------------------------------------------------------------------------------------------------------------------------------------------|
| Global Flag     | W    | whether the customer has global offices                                                                                                     | 
| Major Flag      | W    | whether the customer is a large consumer in their industry (as opposed to SMC - Small Medium Corporation - or SMB - Small Medium Business)  |
| SMC Flag        | W    | whether the customer is a Small Medium Corporation (SMC, as opposed to major and SMB)                                                       |
| Commercial Flag | W    | whether the customer's business is commercial (as opposed to public secor)                                                                  |
| IT Spend        | W    | $ spent on IT-related purchases                                                                                                             |
| Employee Count  | W    | number of employees                                                                                                                         |
| PC Count        | W    | number of PCs used by the customer                                                                                                          |                                                                                      |
| Discount        | T    | whether the customer was given a discount (binary)                                                                                          |
| Revenue         | Y    | $ Revenue from customer given by the amount of software purchased                                                                           |


In [2]:
# Import the sample multi-attribution data
data = (
    spark.read.format("csv")
    .option("inferSchema", True)
    .option("header", True)
    .load(
        "wasbs://publicwasb@mmlspark.blob.core.windows.net/multi_attribution_sample.csv"
    )
)

## Get Causal Effects with SynapseML DoubleMLEstimator

In [ ]:
from synapse.ml.causal import *
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.regression import LinearRegression

treatmentColumn = "Discount"
outcomeColumn = "Revenue"

dml = (
    DoubleMLEstimator()
    .setTreatmentModel(LogisticRegression())
    .setTreatmentCol(treatmentColumn)
    .setOutcomeModel(LinearRegression())
    .setOutcomeCol(outcomeColumn)
    .setMaxIter(20)
)

model = dml.fit(data)

In [7]:
# Get average treatment effect, it returns a numeric value, e.g. 5166.78324
# It means, on average, customers who received a discount spent $5,166 more on software
model.getAvgTreatmentEffect()

In [8]:
# Get treatment effect's confidence interval, e.g.  [4765.826181160708, 5371.2817538168965]
model.getConfidenceInterval()

## Simultaneous continuous treatments (synthetic example)

The plural API estimates an ordered vector of constant, partial-linear treatment
coefficients in one joint final regression. Here the outcome and both treatments
are synthetic and unitless. They are **not** the sample's binary `Discount` or
observed `Revenue`, and these coefficients are not evidence about the startup.
We use only the public sample's population count to choose the simulation size.

We generate two continuous treatments with confounding by observed features
`x` and `z`, and an outcome with known coefficients 2 and -1.5. Independent noise
makes the treatment residuals identifiable in this simulated design. Applying
this method to real data additionally requires credible adjustment for
confounding, temporal ordering, and a suitable partial-linear model; numerical
guards do not establish causal identification.

The seed below controls only the synthetic data. Spark cross-fit splits and
bootstrap sampling are unseeded, so fitted results can vary between runs.


In [ ]:
import numpy as np
from pyspark.ml.regression import LinearRegression
from synapse.ml.causal import DoubleMLEstimator, DoubleMLModel

rng = np.random.default_rng(2328)
# The original public-data example above still runs unchanged.
n_synthetic = data.count()
x = rng.normal(size=n_synthetic)
z = rng.normal(size=n_synthetic)
t1 = 0.8 * x + rng.normal(size=n_synthetic)
t2 = -0.4 * x + 0.3 * z + rng.normal(size=n_synthetic)
y = 2.0 * t1 - 1.5 * t2 + 0.7 * x - 0.5 * z + rng.normal(size=n_synthetic)
synthetic = spark.createDataFrame(
    [
        (float(a), float(b), float(c), float(d), float(e))
        for a, b, c, d, e in zip(x, z, t1, t2, y)
    ],
    ["x", "z", "T1", "T2", "Y"],
)

plural_dml = (
    DoubleMLEstimator()
    .setTreatmentCols(["T1", "T2"])
    .setTreatmentModel(LinearRegression(regParam=0.0))
    .setOutcomeCol("Y")
    .setOutcomeModel(LinearRegression(regParam=0.0))
    .setMaxIter(30)
)
plural_model = plural_dml.fit(synthetic)

In [ ]:
names = plural_model.getFittedTreatmentCols()
means = plural_model.getAvgTreatmentEffects()
intervals = plural_model.getConfidenceIntervals()
p_values = plural_model.getPValues()
draw_rows = plural_model.getRawTreatmentEffectRows()
flat_draws = plural_model.getRawTreatmentEffectsByIteration()
assert names == ["T1", "T2"]
assert len(draw_rows) == 30 and all(len(row) == 2 for row in draw_rows)
assert flat_draws == [value for row in draw_rows for value in row]
for name, mean, interval, p_value in zip(names, means, intervals, p_values):
    print(name, "bootstrap mean:", mean, "95% interval:", interval, "Wald p:", p_value)
# Finite-sample estimates fluctuate around the known synthetic coefficients.

### What the estimates and intervals mean

* The point estimate for each coordinate is the mean of the bootstrap draws,
  not a separate full-sample fit. Each draw first splits original observations
  into two folds, then bootstraps within each fold. All nuisance models in a
  fold use the same sampled rows. Outcome and **all** treatment columns are
  excluded from nuisance features; treatments require an explicit regressor.
  Do not set both `treatmentCol` and `treatmentCols`, and do not pass `weightCol`
  on the plural route.
* The final stage fits a Gaussian generalized linear regression **with an
  intercept in each fold**, then averages the fold coefficient vectors. This
  differs from EconML's pooled, no-intercept final fit; inference parity is not
  claimed.
* Intervals are the coordinate-wise 2.5th and 97.5th bootstrap percentiles. The
  plural p-value is an approximate two-sided normal-bootstrap Wald test using
  the **sample standard deviation of the draws** (divisor B - 1), not that
  spread divided by the square root of B. At least two draws are required;
  zero or nonfinite spread gives `NaN` for that coordinate. There is no
  multiplicity correction or joint coverage. With B = 30, standard errors are
  rough and percentile tails are coarse.
* Relative residual-variance and eigenvalue-ratio guards are numerical policies,
  not identification tests. Extreme scaling can still overflow or underflow.
  Any failed plural draw fails the fit; other already-running draws are not
  promised to be cancelled. Plural-owned caches are cleaned up, but existing
  shared trainers may retain caches on their own failure paths.

The existing singular API and example are unchanged. In particular, singular
bootstrap draws still resample before splitting, so copies of one original row
can cross folds. Singular `getPValue()` retains its existing t-test of the draw
mean, which divides draw spread by the square root of B; it is not the plural
Wald inference above. These are legacy limitations, not guarantees inherited
by the new API.

### Fitted order and Python parameter state

Use the plural convenience getters on a plural fit and the singular getters on
a singular fit. The fitted names and stored width are persisted Spark params,
not immutable objects. The JVM plural getters reject inconsistent current
`treatmentCols`, fitted names, or stored width. Python `setParams`/setters only
change Python parameter state until transferred to the JVM; the convenience
methods below directly query the JVM and therefore report its fitted order.
Generated raw parameter getters on an unset or wrong route may raise `KeyError`;
legacy singular getters are not given new route guards by this change.


In [ ]:
import tempfile

# Save/load must preserve statistics themselves: transform is a passthrough.
with tempfile.TemporaryDirectory() as directory:
    model_path = directory + "/plural-model"
    plural_model.write().save(model_path)
    reloaded = DoubleMLModel.load(model_path)
    assert reloaded.getFittedTreatmentCols() == names
    assert reloaded.getRawTreatmentEffectRows() == draw_rows
    assert reloaded.getAvgTreatmentEffects() == means
    assert reloaded.getConfidenceIntervals() == intervals
    np.testing.assert_allclose(reloaded.getPValues(), p_values, equal_nan=True)